[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/mohsennasab/Rang/blob/main/examples/rang_hydrology_maps_colab.ipynb)

# Hydrologic and climate maps with Rang

This notebook builds six finished figures: Mississippi River basin topography, the river network of the contiguous United States, the early movement of Hurricane Helene's hourly precipitation, October 2021 temperature, May 2022 precipitation, and a 1% annual-chance flood-depth map.

Copernicus DEM GLO-90, NOAA nClimGrid, and supporting boundaries are read through public services. The reduced depth sample is stored in Rang. No upload is needed. The first elevation read can take several minutes because the continental map intersects many DEM tiles.

Created by [Mohsen Tahmasebi Nasab, PhD](https://hydromohsen.com/).

## 1. Set up the notebook

Run this cell once. The figures request Arial. Colab normally uses Liberation Sans when Arial is not installed. Liberation Sans has nearly the same proportions and keeps the layout consistent without distributing a proprietary font.

In [ ]:
%pip install -q "rang-palettes[plots]"             pystac pystac-client planetary-computer odc-stac rioxarray             geopandas pyogrio cartopy rasterio xarray zarr s3fs requests

In [ ]:
import copy
import shutil
import zipfile
from pathlib import Path

import cartopy.crs as ccrs
import geopandas as gpd
import matplotlib as mpl
import matplotlib.pyplot as plt
import numpy as np
import odc.stac
import planetary_computer
import pystac
import pystac_client
import rasterio
import requests
import rioxarray
import s3fs
import xarray as xr
from matplotlib.colors import (
    LightSource,
    LinearSegmentedColormap,
    Normalize,
    PowerNorm,
)
from matplotlib.patches import FancyBboxPatch
from PIL import Image, ImageOps
from rasterio.features import geometry_mask
from rasterio.transform import from_bounds
from shapely.geometry import mapping

import rang

OUTPUT_DIR = Path("/content/rang_maps")
CACHE_DIR = Path("/content/rang_data")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
CACHE_DIR.mkdir(parents=True, exist_ok=True)

mpl.rcParams.update({
    "font.family": "sans-serif",
    "font.sans-serif": ["Arial", "Liberation Sans", "DejaVu Sans"],
    "font.size": 10,
    "figure.facecolor": "white",
    "axes.facecolor": "white",
    "savefig.facecolor": "white",
    "savefig.bbox": "tight",
})

STAC_URL = "https://planetarycomputer.microsoft.com/api/stac/v1"
catalog = pystac_client.Client.open(
    STAC_URL,
    modifier=planetary_computer.sign_inplace,
)

print("Rang palettes:", ", ".join(rang.list_palettes()))
print("Figures will be saved in", OUTPUT_DIR)

### Settings

The DEM resolutions are working resolutions in degrees. They preserve broad terrain while keeping the national examples practical in Colab. Higher hillshade exaggeration strengthens landform without changing elevation values. The climate maps use percentile limits so isolated extremes do not flatten the rest of each palette.

In [ ]:
MISSISSIPPI_DEM_RESOLUTION = 0.0125
TOPOGRAPHY_EXAGGERATION = 7.0
AORC_COARSEN = 4
PRECIPITATION_MIN = 0.1
PRECIPITATION_MAX = 75.0
FIGURE_DPI = 1000

In [ ]:
def download_file(url, destination):
    destination = Path(destination)
    if destination.exists():
        return destination
    destination.parent.mkdir(parents=True, exist_ok=True)
    with requests.get(url, stream=True, timeout=120) as response:
        response.raise_for_status()
        with destination.open("wb") as handle:
            for chunk in response.iter_content(chunk_size=1024 * 1024):
                if chunk:
                    handle.write(chunk)
    return destination


def transparent_cmap(cmap):
    result = copy.copy(cmap)
    result.set_bad((1, 1, 1, 0))
    return result


PALETTE_ART_URLS = {
    "Kashan": "https://raw.githubusercontent.com/mohsennasab/Rang/main/sources/kashan/card.jpg",
    "Termeh": "https://raw.githubusercontent.com/mohsennasab/Rang/main/sources/termeh/cloth.jpg",
    "Shahnameh": "https://raw.githubusercontent.com/mohsennasab/Rang/main/sources/shahnameh/folio.jpg",
    "Rostan": "https://raw.githubusercontent.com/mohsennasab/Rang/main/sources/rostan/growing-in-thus-way.jpg",
    "Mina": "https://raw.githubusercontent.com/mohsennasab/Rang/main/sources/mina/piece.jpg",
}
RANG_LOGO_URL = (
    "https://raw.githubusercontent.com/mohsennasab/Rang/main/"
    "logo/rang_pixel_medallion_logo.png"
)


def add_rang_signature(fig, palette_name):
    art_url = PALETTE_ART_URLS[palette_name]
    art_path = download_file(
        art_url,
        CACHE_DIR / "brand" / f"{palette_name.lower()}-artwork.jpg",
    )
    logo_path = download_file(
        RANG_LOGO_URL,
        CACHE_DIR / "brand" / "rang-logo.png",
    )
    with Image.open(art_path) as source:
        artwork = ImageOps.fit(
            source.convert("RGB"),
            (360, 360),
            method=Image.Resampling.LANCZOS,
        )
    with Image.open(logo_path) as source:
        logo = source.convert("RGBA")

    panel = fig.add_axes([0.755, 0.008, 0.235, 0.092], zorder=100)
    panel.set_xlim(0, 1)
    panel.set_ylim(0, 1)
    panel.set_axis_off()
    panel.add_patch(FancyBboxPatch(
        (0, 0),
        1,
        1,
        boxstyle="round,pad=0.015",
        transform=panel.transAxes,
        facecolor="white",
        edgecolor="#d7d7d3",
        linewidth=0.7,
    ))

    art_axis = panel.inset_axes([0.035, 0.10, 0.25, 0.80])
    art_axis.imshow(artwork)
    art_axis.set_axis_off()

    logo_axis = panel.inset_axes([0.305, 0.19, 0.19, 0.62])
    logo_axis.imshow(logo)
    logo_axis.set_axis_off()

    panel.text(
        0.52,
        0.71,
        f"{palette_name} palette",
        fontsize=8.2,
        weight="bold",
        color="#252525",
        va="center",
    )
    panel.text(
        0.52,
        0.45,
        "Rang",
        fontsize=7.6,
        weight="bold",
        color="#3f3f3f",
        va="center",
    )
    panel.text(
        0.52,
        0.23,
        "github.com/mohsennasab/Rang",
        fontsize=5.4,
        color="#666666",
        va="center",
    )


def finish_map(fig, filename, credit, palette_name):
    fig.text(0.01, 0.008, credit, fontsize=7, color="#595959")
    add_rang_signature(fig, palette_name)
    path = OUTPUT_DIR / filename
    fig.savefig(path, dpi=FIGURE_DPI, bbox_inches="tight", pad_inches=0.08)
    plt.show()
    plt.close(fig)
    print("Saved", path)
    return path


def vertical_endpoint_colorbar(
    ax,
    image,
    title,
    low_label="Low",
    high_label="High",
    position=(0.055, 0.075, 0.025, 0.28),
):
    x, y, width, height = position
    panel = FancyBboxPatch(
        (x - 0.025, y - 0.03),
        0.145,
        height + 0.095,
        transform=ax.transAxes,
        boxstyle="round,pad=0.012",
        facecolor="white",
        edgecolor="none",
        alpha=0.92,
        zorder=20,
    )
    ax.add_patch(panel)
    color_axis = ax.inset_axes(position, zorder=21)
    colorbar = ax.figure.colorbar(image, cax=color_axis, orientation="vertical")
    colorbar.set_ticks([image.norm.vmin, image.norm.vmax])
    colorbar.set_ticklabels([low_label, high_label])
    colorbar.ax.set_title(title, loc="left", fontsize=9, weight="bold", pad=6)
    colorbar.outline.set_visible(False)
    colorbar.ax.tick_params(length=0, pad=4, labelsize=8.5)
    return colorbar


def horizontal_endpoint_colorbar(fig, image, axes, low_label, high_label):
    colorbar = fig.colorbar(
        image,
        ax=axes,
        orientation="horizontal",
        shrink=0.28,
        fraction=0.035,
        aspect=24,
        pad=0.025,
    )
    colorbar.set_ticks([image.norm.vmin, image.norm.vmax])
    colorbar.set_ticklabels([low_label, high_label])
    colorbar.outline.set_visible(False)
    colorbar.ax.tick_params(length=0, pad=4)
    return colorbar


def map_extent(bbox):
    west, south, east, north = bbox
    return [west, east, south, north]


def raster_extent(data):
    return [
        float(data.x.min()),
        float(data.x.max()),
        float(data.y.min()),
        float(data.y.max()),
    ]


def normalize_xy(data):
    if "time" in data.dims:
        data = data.max("time", skipna=True)
    rename = {}
    if "longitude" in data.dims:
        rename["longitude"] = "x"
    if "latitude" in data.dims:
        rename["latitude"] = "y"
    if rename:
        data = data.rename(rename)
    if float(data.x[0]) > float(data.x[-1]):
        data = data.sortby("x")
    if float(data.y[0]) < float(data.y[-1]):
        data = data.sortby("y", ascending=False)
    return data


def clip_raster(data, geometry):
    west, east, south, north = raster_extent(data)
    transform = from_bounds(west, south, east, north, data.sizes["x"], data.sizes["y"])
    keep = geometry_mask(
        [mapping(geometry)],
        out_shape=(data.sizes["y"], data.sizes["x"]),
        transform=transform,
        invert=True,
    )
    return data.where(keep)


def load_pc_dem(name, bbox, resolution, geometry=None, output_crs="EPSG:4326"):
    resolution_label = str(resolution).replace(".", "p")
    crs_label = output_crs.replace(":", "").lower()
    cache_path = CACHE_DIR / f"{name}_{resolution_label}_{crs_label}.npz"
    if cache_path.exists():
        saved = np.load(cache_path)
        return xr.DataArray(
            saved["values"],
            coords={"y": saved["y"], "x": saved["x"]},
            dims=("y", "x"),
        )

    search_args = {"collections": ["cop-dem-glo-90"]}
    if geometry is None:
        search_args["bbox"] = bbox
    else:
        search_args["intersects"] = mapping(geometry)
    items = list(catalog.search(**search_args).items())
    if not items:
        raise ValueError("No Copernicus DEM tiles overlap the requested area")
    print(f"Reading {len(items):,} Copernicus DEM tiles for {name}")
    dataset = odc.stac.load(
        items,
        bands=["data"],
        bbox=bbox,
        crs=output_crs,
        resolution=resolution,
        resampling="bilinear",
        groupby="solar_day",
        fail_on_error=False,
    )
    data = normalize_xy(dataset["data"]).compute()
    np.savez_compressed(
        cache_path,
        values=np.asarray(data.values, dtype=np.float32),
        x=np.asarray(data.x.values),
        y=np.asarray(data.y.values),
    )
    return data


def relief(data, display_mask, exaggeration, horizontal_unit_m):
    values = np.asarray(data.values, dtype=float)
    filled = np.where(np.isfinite(values), values, np.nanmedian(values))
    dx = abs(float(data.x[1] - data.x[0])) * horizontal_unit_m
    dy = abs(float(data.y[1] - data.y[0])) * horizontal_unit_m
    shade = LightSource(azdeg=315, altdeg=34).hillshade(
        filled,
        vert_exag=exaggeration,
        dx=dx,
        dy=dy,
    )
    return np.ma.masked_where(~np.isfinite(display_mask), shade)


def percentile_limits(data, lower=2, upper=98):
    values = np.asarray(data, dtype=float)
    return np.nanpercentile(values, [lower, upper])

### U.S. and Mississippi River basin boundaries

Census state boundaries define the contiguous United States. HydroBASINS supplies one transboundary Mississippi basin polygon that includes the small Canadian portion north of the border.

In [ ]:
states_zip = download_file(
    "https://www2.census.gov/geo/tiger/GENZ2024/shp/cb_2024_us_state_20m.zip",
    CACHE_DIR / "cb_2024_us_state_20m.zip",
)
states = gpd.read_file(f"zip://{states_zip}").to_crs("EPSG:4326")
excluded = {"AK", "HI", "PR", "VI", "GU", "MP", "AS"}
conus_states = states.loc[~states["STUSPS"].isin(excluded)].copy()
conus_geometry = conus_states.geometry.union_all()
CONUS_BBOX = (-125.0, 24.0, -66.5, 50.0)

HYDROBASINS_URL = (
    "https://data.hydrosheds.org/file/hydrobasins/standard/"
    "hybas_na_lev03_v1c.zip"
)
basins_zip = download_file(HYDROBASINS_URL, CACHE_DIR / "hybas_na_lev03_v1c.zip")
basins_folder = CACHE_DIR / "hybas_na_lev03_v1c"
basin_shapefile = basins_folder / "hybas_na_lev03_v1c.shp"
if not basin_shapefile.exists():
    with zipfile.ZipFile(basins_zip) as archive:
        archive.extractall(basins_folder)
continental_basins = gpd.read_file(basin_shapefile)
mississippi_basin = continental_basins.loc[
    continental_basins["HYBAS_ID"].eq(7030047060), "geometry"
].iloc[0]
if not mississippi_basin.is_valid:
    mississippi_basin = mississippi_basin.buffer(0)
MISSISSIPPI_BBOX = (-115.0, 28.0, -77.0, 51.5)
MISSISSIPPI_MAP_BBOX = (-125.0, 24.0, -66.5, 51.5)

basin_states = conus_states.loc[
    conus_states.geometry.intersects(mississippi_basin)
].copy()
basin_state_labels = basin_states.copy()
basin_state_labels["geometry"] = basin_state_labels.geometry.intersection(
    mississippi_basin
)
basin_state_labels["label_point"] = basin_state_labels.geometry.representative_point()

usa_projection = ccrs.AlbersEqualArea(
    central_longitude=-96,
    central_latitude=38,
    standard_parallels=(29.5, 45.5),
)
print("Boundaries are ready")

## 2. Mississippi River basin topography with inverted Kashan

Elevation comes from [Copernicus DEM GLO-90 on Microsoft Planetary Computer](https://planetarycomputer.microsoft.com/dataset/cop-dem-glo-90). The full contiguous United States remains visible around the transboundary basin. A stronger hillshade gives the terrain a clear three-dimensional form. A power stretch devotes more of Kashan to the low-lying basin, revealing subtle relief along the Mississippi corridor instead of compressing it into one color.

In [ ]:
mississippi_dem = load_pc_dem(
    "cop_dem_mississippi",
    MISSISSIPPI_BBOX,
    MISSISSIPPI_DEM_RESOLUTION,
    geometry=mississippi_basin,
)
basin_dem = clip_raster(mississippi_dem, mississippi_basin)
basin_values = np.asarray(basin_dem.values, dtype=float)
elevation_min, elevation_max = percentile_limits(basin_values, 0.5, 99.5)
elevation_min = max(0, elevation_min)
elevation_norm = PowerNorm(
    gamma=0.48,
    vmin=elevation_min,
    vmax=elevation_max,
    clip=True,
)
elevation_cmap = transparent_cmap(rang.cmap("Kashan", direction=-1))
basin_shade = relief(
    mississippi_dem,
    basin_values,
    TOPOGRAPHY_EXAGGERATION,
    horizontal_unit_m=111_320,
)

In [ ]:
fig = plt.figure(figsize=(11.2, 8.2))
ax = fig.add_subplot(1, 1, 1, projection=usa_projection)
ax.set_extent(map_extent(MISSISSIPPI_MAP_BBOX), crs=ccrs.PlateCarree())
ax.set_facecolor("white")
image = ax.imshow(
    np.ma.masked_invalid(basin_values),
    extent=raster_extent(basin_dem),
    origin="upper",
    transform=ccrs.PlateCarree(),
    cmap=elevation_cmap,
    norm=elevation_norm,
    interpolation="bilinear",
    zorder=1,
)
ax.imshow(
    basin_shade,
    extent=raster_extent(mississippi_dem),
    origin="upper",
    transform=ccrs.PlateCarree(),
    cmap="gray",
    vmin=0,
    vmax=1,
    alpha=0.3,
    interpolation="bilinear",
    zorder=2,
)
ax.add_geometries(
    conus_states.geometry,
    ccrs.PlateCarree(),
    facecolor="none",
    edgecolor="#6f7375",
    linewidth=0.38,
    alpha=0.48,
    zorder=3,
)
for _, state in basin_state_labels.iterrows():
    point = state["label_point"]
    ax.text(
        point.x,
        point.y,
        state["NAME"],
        transform=ccrs.PlateCarree(),
        ha="center",
        va="center",
        fontsize=5.8,
        color="#5f6466",
        alpha=0.58,
        family="Arial",
        zorder=4,
    )
ax.add_geometries(
    [mississippi_basin],
    ccrs.PlateCarree(),
    facecolor="none",
    edgecolor="#303030",
    linewidth=0.65,
    zorder=5,
)
ax.set_axis_off()
vertical_endpoint_colorbar(
    ax,
    image,
    "Elevation",
    "Low elevation",
    "High elevation",
)
finish_map(
    fig,
    "mississippi_topography_kashan.png",
    "Elevation: Copernicus DEM GLO-90 via Microsoft Planetary Computer. Basin: HydroBASINS.",
    "Kashan",
)

## 3. Rivers of the contiguous United States with Termeh

This section retains every HydroRIVERS reach that intersects the contiguous United States. Large downstream reaches are darker and wider. Minor reaches remain fine enough to read as a national network. A quiet CONUS outline holds the network on the page without adding internal state boundaries.

In [ ]:
HYDRORIVERS_URL = "https://data.hydrosheds.org/file/HydroRIVERS/HydroRIVERS_v10_na_shp.zip"
rivers_zip = download_file(HYDRORIVERS_URL, CACHE_DIR / "HydroRIVERS_v10_na_shp.zip")
rivers_folder = CACHE_DIR / "hydrorivers_north_america"
river_shapefile = rivers_folder / "HydroRIVERS_v10_na_shp" / "HydroRIVERS_v10_na.shp"
if not river_shapefile.exists():
    with zipfile.ZipFile(rivers_zip) as archive:
        archive.extractall(rivers_folder)

rivers = gpd.read_file(
    river_shapefile,
    engine="pyogrio",
    bbox=CONUS_BBOX,
    columns=["ORD_FLOW", "DIS_AV_CMS"],
)
rivers = gpd.clip(rivers.loc[rivers.geometry.notna()], conus_geometry)
rivers["ORD_FLOW"] = rivers["ORD_FLOW"].astype(int)
rivers_projected = rivers.to_crs(usa_projection.proj4_init)
rivers_projected["geometry"] = rivers_projected.geometry.simplify(
    400,
    preserve_topology=False,
)
rivers_projected = rivers_projected.sort_values("ORD_FLOW", ascending=False)
print(f"Loaded {len(rivers_projected):,} river reaches inside the contiguous United States")

In [ ]:
river_orders = sorted(rivers_projected["ORD_FLOW"].unique())
river_colors = rang.rang(
    "Termeh",
    len(river_orders) + 2,
    kind="continuous",
    direction=-1,
)[:len(river_orders)]
color_by_order = dict(zip(river_orders, river_colors))
major_order = min(river_orders)
width_by_order = {
    order: max(0.025, 2.8 * (0.55 ** (order - major_order)))
    for order in river_orders
}

fig = plt.figure(figsize=(12.2, 7.8))
ax = fig.add_subplot(1, 1, 1, projection=usa_projection)
ax.set_extent(map_extent(CONUS_BBOX), crs=ccrs.PlateCarree())
ax.set_facecolor("white")
rivers_projected.plot(
    ax=ax,
    color=rivers_projected["ORD_FLOW"].map(color_by_order),
    linewidth=rivers_projected["ORD_FLOW"].map(width_by_order),
    alpha=0.96,
    rasterized=True,
    zorder=1,
)
ax.add_geometries(
    [conus_geometry],
    ccrs.PlateCarree(),
    facecolor="none",
    edgecolor="#a7aaa8",
    linewidth=0.5,
    alpha=0.85,
    zorder=2,
)
ax.set_axis_off()
finish_map(
    fig,
    "usa_rivers_termeh.png",
    "Rivers: HydroRIVERS v1.0, HydroSHEDS. CONUS boundary: U.S. Census Bureau.",
    "Termeh",
)

## 4. Hurricane Helene precipitation with Shahnameh

AORC stores one-hour precipitation ending at each UTC timestamp. These six panels cover 04:00 through 14:00 UTC on September 27, 2024. They share one fixed scale, while zero precipitation and missing cells remain transparent.

In [ ]:
AORC_TIMES_UTC = [
    "2024-09-27T04:00:00",
    "2024-09-27T06:00:00",
    "2024-09-27T08:00:00",
    "2024-09-27T10:00:00",
    "2024-09-27T12:00:00",
    "2024-09-27T14:00:00",
]
HELENE_BBOX = (-92.0, 23.5, -74.0, 39.5)
s3 = s3fs.S3FileSystem(anon=True)
store = s3fs.S3Map(
    root="noaa-nws-aorc-v1-1-1km/2024.zarr",
    s3=s3,
    check=False,
)
aorc = xr.open_zarr(store, consolidated=True)
precipitation = aorc["APCP_surface"].sel(
    time=np.asarray(AORC_TIMES_UTC, dtype="datetime64[ns]")
)
latitude_slice = slice(HELENE_BBOX[1], HELENE_BBOX[3])
if float(aorc.latitude[0]) > float(aorc.latitude[-1]):
    latitude_slice = slice(HELENE_BBOX[3], HELENE_BBOX[1])
precipitation = precipitation.sel(
    latitude=latitude_slice,
    longitude=slice(HELENE_BBOX[0], HELENE_BBOX[2]),
)
if AORC_COARSEN > 1:
    precipitation = precipitation.coarsen(
        latitude=AORC_COARSEN,
        longitude=AORC_COARSEN,
        boundary="trim",
    ).max(skipna=True)
precipitation = precipitation.compute()

rain_values = np.asarray(precipitation.values, dtype=float)
if float(precipitation.latitude[0]) < float(precipitation.latitude[-1]):
    rain_values = rain_values[:, ::-1, :]
rain_values = np.ma.masked_where(
    ~np.isfinite(rain_values) | (rain_values <= 0),
    rain_values,
)
rain_extent = [
    float(precipitation.longitude.min()),
    float(precipitation.longitude.max()),
    float(precipitation.latitude.min()),
    float(precipitation.latitude.max()),
]
print("Maximum across all panels:", f"{float(rain_values.max()):.1f} mm")

In [ ]:
visible_states = conus_states.cx[
    HELENE_BBOX[0]:HELENE_BBOX[2], HELENE_BBOX[1]:HELENE_BBOX[3]
]
rain_cmap = transparent_cmap(rang.cmap("Shahnameh"))
rain_norm = PowerNorm(
    gamma=0.55,
    vmin=PRECIPITATION_MIN,
    vmax=PRECIPITATION_MAX,
    clip=True,
)
helene_projection = ccrs.LambertConformal(
    central_longitude=-83,
    central_latitude=32,
    standard_parallels=(27, 37),
)
fig, axes = plt.subplots(
    2,
    3,
    figsize=(13.0, 8.7),
    subplot_kw={"projection": helene_projection},
)
fig.subplots_adjust(left=0.02, right=0.98, top=0.985, bottom=0.11, wspace=0.025, hspace=0.04)

image = None
for index, ax in enumerate(axes.flat):
    ax.set_extent(map_extent(HELENE_BBOX), crs=ccrs.PlateCarree())
    ax.set_facecolor("white")
    ax.add_geometries(
        visible_states.geometry,
        ccrs.PlateCarree(),
        facecolor="#f3f2ee",
        edgecolor="#aaaaa6",
        linewidth=0.42,
        zorder=1,
    )
    image = ax.imshow(
        rain_values[index],
        extent=rain_extent,
        origin="upper",
        transform=ccrs.PlateCarree(),
        cmap=rain_cmap,
        norm=rain_norm,
        interpolation="nearest",
        zorder=2,
    )
    utc_label = AORC_TIMES_UTC[index][11:16]
    ax.text(
        0.025,
        0.965,
        f"{index + 1}   {utc_label} UTC",
        transform=ax.transAxes,
        ha="left",
        va="top",
        fontsize=10,
        weight="bold",
        color="#222222",
        bbox={
            "boxstyle": "round,pad=0.32",
            "facecolor": "white",
            "edgecolor": "none",
            "alpha": 0.94,
        },
        zorder=4,
    )
    ax.set_axis_off()

horizontal_endpoint_colorbar(
    fig,
    image,
    axes.ravel().tolist(),
    "Low precipitation",
    "High precipitation",
)
finish_map(
    fig,
    "hurricane_helene_sequence_shahnameh.png",
    "Hourly precipitation ending at each time: NOAA AORC v1.1. Boundaries: U.S. Census Bureau.",
    "Shahnameh",
)

## 5. October 2021 temperature with inverted Rostan

This example follows the Microsoft Planetary Computer STAC pattern and opens the `tavg` cloud-optimized GeoTIFF from `nclimgrid-202110`. The asset stores monthly average temperature in degrees Celsius. Inverted Rostan is stretched between the second and ninety-eighth percentiles.

In [ ]:
temperature_item_url = (
    "https://planetarycomputer.microsoft.com/api/stac/v1/collections/"
    "noaa-nclimgrid-monthly/items/nclimgrid-202110"
)
temperature_item = planetary_computer.sign(
    pystac.Item.from_file(temperature_item_url)
)
temperature = rioxarray.open_rasterio(
    temperature_item.assets["tavg"].href,
    masked=True,
).squeeze(drop=True)
temperature = clip_raster(temperature, conus_geometry)
temperature_values = np.asarray(temperature.values, dtype=float)
temperature_min, temperature_max = percentile_limits(temperature_values, 2, 98)
temperature_norm = Normalize(vmin=temperature_min, vmax=temperature_max, clip=True)
temperature_cmap = transparent_cmap(rang.cmap("Rostan", direction=-1))

In [ ]:
fig = plt.figure(figsize=(12.2, 7.8))
ax = fig.add_subplot(1, 1, 1, projection=usa_projection)
ax.set_extent(map_extent(CONUS_BBOX), crs=ccrs.PlateCarree())
ax.set_facecolor("white")
image = ax.imshow(
    np.ma.masked_invalid(temperature_values),
    extent=raster_extent(temperature),
    origin="upper",
    transform=ccrs.PlateCarree(),
    cmap=temperature_cmap,
    norm=temperature_norm,
    interpolation="bilinear",
    zorder=1,
)
ax.add_geometries(
    conus_states.geometry,
    ccrs.PlateCarree(),
    facecolor="none",
    edgecolor="#5f5f5f",
    linewidth=0.28,
    alpha=0.75,
    zorder=2,
)
ax.set_axis_off()
vertical_endpoint_colorbar(ax, image, "Temperature", "Cooler", "Warmer")
finish_map(
    fig,
    "usa_temperature_rostan.png",
    "Monthly average temperature, October 2021: NOAA nClimGrid via Microsoft Planetary Computer.",
    "Rostan",
)

## 6. May 2022 precipitation with Mina

The `prcp` asset from `nclimgrid-202205` stores monthly precipitation in millimeters. A continuous power stretch gives more of Mina to the lower end of the distribution, making dry and moderately wet regions easier to distinguish without breaking the surface into classes.

In [ ]:
precipitation_item_url = (
    "https://planetarycomputer.microsoft.com/api/stac/v1/collections/"
    "noaa-nclimgrid-monthly/items/nclimgrid-202205"
)
precipitation_item = planetary_computer.sign(
    pystac.Item.from_file(precipitation_item_url)
)
monthly_precipitation = rioxarray.open_rasterio(
    precipitation_item.assets["prcp"].href,
    masked=True,
).squeeze(drop=True)
monthly_precipitation = clip_raster(monthly_precipitation, conus_geometry)
monthly_precipitation_values = np.asarray(monthly_precipitation.values, dtype=float)
monthly_valid = monthly_precipitation_values[
    np.isfinite(monthly_precipitation_values)
]
monthly_min, monthly_max = percentile_limits(monthly_valid, 0.5, 99.5)
monthly_min = max(0, monthly_min)
monthly_norm = PowerNorm(
    gamma=0.45,
    vmin=monthly_min,
    vmax=monthly_max,
    clip=True,
)
monthly_cmap = transparent_cmap(rang.cmap("Mina"))

In [ ]:
fig = plt.figure(figsize=(12.2, 7.8))
ax = fig.add_subplot(1, 1, 1, projection=usa_projection)
ax.set_extent(map_extent(CONUS_BBOX), crs=ccrs.PlateCarree())
ax.set_facecolor("white")
image = ax.imshow(
    np.ma.masked_invalid(monthly_precipitation_values),
    extent=raster_extent(monthly_precipitation),
    origin="upper",
    transform=ccrs.PlateCarree(),
    cmap=monthly_cmap,
    norm=monthly_norm,
    interpolation="bilinear",
    zorder=1,
)
ax.add_geometries(
    conus_states.geometry,
    ccrs.PlateCarree(),
    facecolor="none",
    edgecolor="#4d4d4d",
    linewidth=0.28,
    alpha=0.72,
    zorder=2,
)
ax.set_axis_off()
vertical_endpoint_colorbar(ax, image, "Precipitation", "Low", "High")
finish_map(
    fig,
    "usa_precipitation_mina.png",
    "Monthly precipitation, May 2022: NOAA nClimGrid via Microsoft Planetary Computer.",
    "Mina",
)

## 7. Whiskey Chitto 1% flood depth with Termeh

The reduced `BLE_DEP01PCT` GeoTIFF stored in Rang provides flood depth in feet. Termeh carries the depth values over a clean white background. This remains a visual example, not an engineering product.

In [ ]:
DEPTH_URL = (
    "https://raw.githubusercontent.com/mohsennasab/Rang/main/"
    "data/whiskey_chitto_depth_1pct.tif"
)
depth_path = download_file(
    DEPTH_URL,
    CACHE_DIR / "whiskey_chitto_depth_1pct.tif",
)
depth_source = rioxarray.open_rasterio(depth_path, masked=True).squeeze(drop=True)
depth_geo = depth_source.rio.reproject(
    "EPSG:4326",
    resolution=0.001,
    resampling=rasterio.enums.Resampling.bilinear,
)
depth = np.ma.masked_invalid(np.asarray(depth_geo.values, dtype=float))
depth_min, depth_max = percentile_limits(depth.compressed(), 1, 99)
depth_min = max(0, depth_min)
depth_norm = PowerNorm(
    gamma=0.65,
    vmin=depth_min,
    vmax=depth_max,
    clip=True,
)
depth_colors = rang.rang("Termeh", 9, kind="continuous")[1:]
depth_cmap = transparent_cmap(
    LinearSegmentedColormap.from_list("Termeh depth", depth_colors)
)

depth_lonlat_bbox = (
    float(depth_geo.x.min()),
    float(depth_geo.y.min()),
    float(depth_geo.x.max()),
    float(depth_geo.y.max()),
)

In [ ]:
fig, ax = plt.subplots(figsize=(10.2, 8.4))
ax.set_facecolor("white")
image = ax.imshow(
    depth,
    extent=raster_extent(depth_geo),
    origin="upper",
    cmap=depth_cmap,
    norm=depth_norm,
    interpolation="bilinear",
    zorder=1,
)
ax.set_xlim(depth_lonlat_bbox[0], depth_lonlat_bbox[2])
ax.set_ylim(depth_lonlat_bbox[1], depth_lonlat_bbox[3])
ax.set_aspect("equal")
ax.set_axis_off()
vertical_endpoint_colorbar(ax, image, "Depth", "Low", "High")
finish_map(
    fig,
    "whiskey_chitto_depth_termeh.png",
    "Flood depth: reduced BLE_DEP01PCT raster.",
    "Termeh",
)

## 8. Download the finished figures

This cell collects every PNG created during the session. It is fine to skip a section. Only files that exist are included.

In [ ]:
from google.colab import files

png_files = sorted(OUTPUT_DIR.glob("*.png"))
if not png_files:
    raise ValueError("Run at least one map section before downloading")

archive_base = Path("/content/rang_hydrology_maps")
archive_path = Path(shutil.make_archive(str(archive_base), "zip", OUTPUT_DIR))
print("Included files")
for path in png_files:
    print(" ", path.name)
files.download(str(archive_path))

## Data sources

- [Copernicus DEM GLO-90 on Microsoft Planetary Computer](https://planetarycomputer.microsoft.com/dataset/cop-dem-glo-90)
- [NOAA nClimGrid monthly on Microsoft Planetary Computer](https://planetarycomputer.microsoft.com/dataset/noaa-nclimgrid-monthly)
- [HydroBASINS from HydroSHEDS](https://www.hydrosheds.org/products/hydrobasins)
- [HydroRIVERS from HydroSHEDS](https://www.hydrosheds.org/products/hydrorivers)
- [NOAA Analysis of Record for Calibration](https://registry.opendata.aws/noaa-nws-aorc/)
- [U.S. Census Bureau cartographic boundary files](https://www.census.gov/geographies/mapping-files/time-series/geo/cartographic-boundary.html)
- [Whiskey Chitto flood-depth sample metadata](../data/whiskey_chitto_depth_1pct.json)

Review each source's documentation and terms before publishing or redistributing derived work.